# Grand Hyrax Run Script Generator

This notebook is the main workflow for generating Hyrax runtime TOMLs and Delta SLURM `.sh` files.

Experiment numbers are per run. If `run1` is `HyraxAutoencoderV2` and `run2` is `SimCLR`, both runs can use suffixes `_1`, `_2`, `_3`, etc. That means SimCLR does not need `_101` style experiment numbers unless you explicitly choose that yourself.

In [ ]:
from dataclasses import replace
from pathlib import Path

from grand_run_script_generator import (
    BASELINE_EXPERIMENT_NUMBER,
    OPTIMIZER_GRID_V2,
    build_autoencoder_variant_specs,
    build_extra_experiment_specs,
    build_optimizer_sweep_specs,
    configured_run_plan,
    create_3d_viz_json_batch,
    create_3dumap_scripts_batch,
    create_infer_scripts_batch,
    create_training_files,
    create_udb_scripts_batch,
    discover_builtin_model_names,
    optimizer_config,
    parse_number_spec,
    print_experiment_specs,
    submit_jobs,
)


## Available Models

In [ ]:
discover_builtin_model_names()


## Configure Runs

Set one model per run number. Keep experiment suffixes simple inside each run, usually `_1` through `_8` for the optimizer grid and optional `_9` onward for variants.

In [ ]:
PROFILE = "delta"  # Use "local" for smoke tests on your workstation.

# Set this to None for all experiments in each selected run, or strings like "1-8" / "1-8,11,18".
DEFAULT_EXPERIMENT_NUMBERS = None

# Optional path overrides. Leave empty to use research_paths.py and HYRAX_PROFILE/PROFILE.
PATH_OVERRIDES = {
    # "base_directory": "/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs",
    # "data_dir": "/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/split_images_120/",
    # "results_dir": "/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/results/",
    # "filter_catalog": "/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/split_images/catalog2.fits",
}

RUNS = {
    1: {
        "model_name": "HyraxAutoencoderV2",
        "include_autoencoder_variants": True,
        "baseline_experiment_number": 7,
        "crop_to": [120, 120],
        "batch_size": 256,
        "epochs": 20,
    },
    # 2: {
    #     "model_name": "SimCLR",
    #     "filters": ["g", "r", "i"],
    #     "crop_to": [120, 120],
    #     "batch_size": 256,
    #     "epochs": 20,
    #     "experiments": "1-8",
    # },
    # 3: {
    #     "model_name": "ImageDCAE",
    #     "crop_to": [120, 120],
    #     "batch_size": 256,
    #     "epochs": 20,
    #     "experiments": "1-8",
    # },
}

# Pick which run numbers to operate on. Use sorted(RUNS) for all configured runs.
SELECTED_RUNS = sorted(RUNS)


## Optional Custom Experiments

Add as many experiments as you want by putting `extra_experiments` inside a run config. These inherit optimizer/settings from a baseline when `baseline_experiment` is provided.

In [ ]:
# Example snippets to paste into a RUNS entry:
#
# "optimizer_grid": {
#     1: optimizer_config("torch.optim.SGD", 0.01, momentum=True),
#     2: optimizer_config("torch.optim.Adam", 0.001),
# },
#
# "extra_experiments": {
#     9: {
#         "description": "baseline _7 with epochs=40",
#         "baseline_experiment": 7,
#         "overrides": {"epochs": 40},
#     },
#     10: {
#         "description": "latent_dim=256",
#         "baseline_experiment": 7,
#         "overrides": {"model_config_overrides": {"HyraxAutoencoderV2": {"latent_dim": 256}}},
#     },
# }


## Build Plans

In [ ]:
def build_specs_for_run(run_config):
    model_name = run_config["model_name"]
    optimizer_grid = run_config.get("optimizer_grid", OPTIMIZER_GRID_V2)
    common_kwargs = {
        "model_name": model_name,
        "start_number": 1,
        "optimizer_grid": optimizer_grid,
        "filters": run_config.get("filters", ["g", "r", "i"] if model_name == "SimCLR" else None),
        "model_config_overrides": run_config.get("model_config_overrides"),
        "overrides": run_config.get("overrides"),
    }
    if run_config.get("include_autoencoder_variants", False):
        specs = build_autoencoder_variant_specs(
            baseline_experiment_number=run_config.get("baseline_experiment_number", BASELINE_EXPERIMENT_NUMBER),
            **common_kwargs,
        )
    else:
        specs = build_optimizer_sweep_specs(**common_kwargs)

    extras = build_extra_experiment_specs(
        run_config.get("extra_experiments"),
        specs,
        default_baseline_experiment=run_config.get("baseline_experiment_number", 1),
    )
    specs.update(extras)
    return dict(sorted(specs.items()))


def build_plan_for_run(run_number, run_config):
    path_kwargs = {
        key: run_config.get(key, PATH_OVERRIDES.get(key))
        for key in ("base_directory", "data_dir", "results_dir", "filter_catalog")
        if run_config.get(key, PATH_OVERRIDES.get(key)) is not None
    }
    plan = configured_run_plan(
        profile=run_config.get("profile", PROFILE),
        run_number=run_number,
        model_name=run_config["model_name"],
        **path_kwargs,
    )

    update_keys = (
        "batch_size",
        "epochs",
        "crop_to",
        "dataset_class",
        "object_id_column_name",
        "filters",
        "transform",
        "data_fields",
        "primary_id_field",
        "data_set",
        "slurm",
        "shell_preamble",
        "setup_lines",
    )
    updates = {key: run_config[key] for key in update_keys if key in run_config}
    return replace(plan, **updates) if updates else plan


def selected_experiments_for_run(run_config, specs):
    requested = run_config.get("experiments", DEFAULT_EXPERIMENT_NUMBERS)
    return parse_number_spec(requested, default=sorted(specs))


RUN_CONTEXT = {}
for run_number in SELECTED_RUNS:
    run_config = RUNS[run_number]
    specs = build_specs_for_run(run_config)
    experiments = selected_experiments_for_run(run_config, specs)
    plan = build_plan_for_run(run_number, run_config)
    RUN_CONTEXT[run_number] = {
        "config": run_config,
        "specs": specs,
        "experiments": experiments,
        "plan": plan,
    }

    print(f"\nrun{run_number}: {run_config['model_name']}")
    print(f"Run directory: {plan.run_dir}")
    print_experiment_specs({idx: specs[idx] for idx in experiments})


## Generate Training TOMLs and SLURM Scripts

In [ ]:
for run_number, ctx in RUN_CONTEXT.items():
    create_training_files(ctx["plan"], ctx["specs"], ctx["experiments"])


## Submit Training Jobs

In [ ]:
DRY_RUN = True  # Set False on Delta when you are ready to submit.

for run_number, ctx in RUN_CONTEXT.items():
    submit_jobs(
        run_number=run_number,
        job_numbers=ctx["experiments"],
        prefix="train",
        base_directory=ctx["plan"].base_directory,
        dry_run=DRY_RUN,
    )


## Generate and Submit Inference Jobs

Run this after training jobs finish and `train*.txt` logs exist.

In [ ]:
for run_number, ctx in RUN_CONTEXT.items():
    create_infer_scripts_batch(ctx["plan"], ctx["experiments"])


In [ ]:
DRY_RUN = True

for run_number, ctx in RUN_CONTEXT.items():
    submit_jobs(run_number, ctx["experiments"], "infer", ctx["plan"].base_directory, dry_run=DRY_RUN)


## Generate and Submit UMAP + Database Jobs

Run this after inference jobs finish and `infer*.txt` logs exist.

In [ ]:
for run_number, ctx in RUN_CONTEXT.items():
    create_udb_scripts_batch(ctx["plan"], ctx["experiments"])


In [ ]:
DRY_RUN = True

for run_number, ctx in RUN_CONTEXT.items():
    submit_jobs(run_number, ctx["experiments"], "udb", ctx["plan"].base_directory, dry_run=DRY_RUN)


## Generate and Submit 3D UMAP Jobs

In [ ]:
for run_number, ctx in RUN_CONTEXT.items():
    create_3dumap_scripts_batch(ctx["plan"], ctx["experiments"])


In [ ]:
DRY_RUN = True

for run_number, ctx in RUN_CONTEXT.items():
    submit_jobs(run_number, ctx["experiments"], "3dumap", ctx["plan"].base_directory, dry_run=DRY_RUN)


## Create 3D Visualization JSON Files

Run this after 3D UMAP jobs finish and `3dumap*.txt` logs exist.

In [ ]:
for run_number, ctx in RUN_CONTEXT.items():
    create_3d_viz_json_batch(ctx["plan"], ctx["experiments"], id_column="object_id")
